# Sàng lọc Bệnh Parkinson Qua Giọng Nói (Đồ án 2.1)
**Voice-Based Parkinson's Disease Screening Using Machine Learning**

Notebook này:
1. Khám phá UCI Parkinson's Voice Dataset và cấu trúc theo bệnh nhân (subject).
2. Huấn luyện & so sánh LR / SVM / Random Forest / Gradient Boosting với chia dữ liệu theo bệnh nhân (chống data leakage).
3. Trích xuất đặc trưng từ audio (Praat + Librosa) và suy luận.

> Hệ thống chỉ phục vụ mục đích nghiên cứu và sàng lọc ban đầu, không phải công cụ chẩn đoán y khoa.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
from audio_project import parkinson as pk

## 1. Khám phá dữ liệu
Mỗi bệnh nhân có ~6 bản ghi nguyên âm /a/. Subject ID nằm trong cột `name` (`phon_R01_S01_1` → `S01`).

In [ ]:
df = pd.read_csv(pk.UCI_CSV_PATH)
df['subject'] = pk.uci_subject_ids(df['name'])
print('Bản ghi:', len(df), '| Subject:', df['subject'].nunique())
print(df.groupby('subject')['status'].first().value_counts().rename({1: 'Parkinson', 0: 'Control'}))
df.groupby('status')[pk.UCI_MODEL_COLS].median().T

## 2. Huấn luyện & so sánh mô hình
- Model chỉ dùng 16 cột MDVP tái tạo được từ audio bằng Praat (F0, Jitter, Shimmer, HNR, NHR).
- Chia train/test bằng `StratifiedGroupKFold` theo subject (~25% subject vào test).
- GridSearchCV theo subject trên tập train → model triển khai, đánh giá trên tập test giữ riêng.
- Nested CV theo subject trên toàn bộ 32 người → ước lượng chính và tiêu chí chọn model.

In [ ]:
result = pk.train_parkinson_model(verbose=True)
pd.read_csv(pk.COMPARISON_CSV_PATH)

In [ ]:
print(result.report)
display(Image(filename=str(ROOT / 'results' / 'confusion_matrix.png')))
display(Image(filename=str(ROOT / 'results' / 'roc_curve.png')))

## 3. Trích xuất đặc trưng & suy luận từ audio
Demo trên tín hiệu nguyên âm tổng hợp (chuỗi xung thanh môn 150 Hz). Thay bằng file WAV ghi từ INMP441 để thử thực tế.

In [ ]:
import soundfile as sf, tempfile

sr = pk.SR
rng = np.random.default_rng(0)
y = np.zeros(2 * sr); t = 0.0
while t < 2:
    y[int(t * sr)] = 1.0
    t += (1 / 150) * (1 + rng.normal(0, 0.003))
y = np.convolve(y, np.exp(-np.arange(64) / 13))[: 2 * sr] + rng.normal(0, 0.005, 2 * sr)

wav = Path(tempfile.mkdtemp()) / 'vowel.wav'
sf.write(wav, y / np.abs(y).max(), sr)

feats = pk.extract_features_from_file(wav)
print(f'{len(feats)} đặc trưng')
for k in ['f0_mean', 'f0_std', 'jitter_local', 'shimmer_local', 'hnr', 'nhr']:
    print(f'{k:>14}: {feats[k]:.5f}')

pk.predict_parkinson(wav)